# GOES-19 ABI Extraction

This notebook demonstrates how to search for GOES-19 ABI Full Disk granules via **AWS S3** and extract a specific band using **satpy**.

## Setup

Import the required libraries and configure the search parameters.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
import time

import geopandas as gpd

from aer.client import AerClient
from aer.interfaces import ExtractionProfile

## Configuration

Define the AOI, time range, and collection.

In [ ]:
# --- Configuration ---
BAND = "C07"  # GOES ABI band
DATE_START = datetime(2026, 4, 1, 15, 0, tzinfo=timezone.utc)
DATE_END = datetime(2026, 4, 1, 15, 10, tzinfo=timezone.utc)

# Load AOI (Buenos Aires province)
geojson_path = Path("buenos_aires.geojson")
gdf = gpd.read_file(geojson_path)
aoi = gdf.geometry.iloc[0]

# GOES-19 ABI Full Disk collection
collections = ["ABI-L1b-RadF"]

# --- Client Setup ---
client = AerClient()

## Search

Search for GOES-19 ABI granules using `search_aws_goes` (public S3, no auth needed).

In [ ]:
print("Searching...", flush=True)
results = client.search(
    collections=collections,
    start_datetime=DATE_START,
    end_datetime=DATE_END,
    intersects=aoi,
    search_params={"ABI-L1b-RadF": {"satellite": "GOES-19"}},
    plugin_hints={"search_aws_goes": collections},
)
print(f"Found {len(results)} results", flush=True)
results.head()

## Prepare Extraction

Define extraction profiles and prepare tasks.

In [ ]:
# --- Prepare Extraction ---
uri = "extract_buenos_aires_goes"

profiles = [
    ExtractionProfile(
        name="goes_band",
        resolution=2000,
        collection_variables_map={"ABI-L1b-RadF": [BAND]},
        extra_params={"reader": "abi_l1b"},
    )
]

prepare_params = {"cells_per_chunk": 10}

tasks = client.prepare_for_extraction(
    results,
    target_aoi=aoi,
    uri=uri,
    profiles=profiles,
    target_grid_dist=256000,
    target_grid_overlap=False,
    prepare_params=prepare_params,
    plugin_hints={"extract_satpy": collections},
)

print(f"Prepared {len(tasks)} extraction tasks", flush=True)

## Extract

Run the extraction pipeline (public S3 access, no downloader needed).

In [ ]:
# Clean output directory
uri_path = Path(uri)
if uri_path.exists():
    shutil.rmtree(uri_path)
uri_path.mkdir(parents=True)

print("Extracting...", flush=True)
start_time = time.time()

extract_params = {
    "padding": 2,
    "resampling": "nearest",
    "calibration": "radiance",
}

results_df = client.extract_batches(
    tasks,
    extract_params=extract_params,
    plugin_hints={"extract_satpy": collections},
    max_batch_workers=4,
)

end_time = time.time()
print(f"Extraction took {end_time - start_time:.2f}s")
print(f"Extracted {len(results_df)} artifacts")

## Results

Inspect the extracted artifacts.

In [ ]:
results_df[["id", "collection", "grid_cell", "uri"]].head()